# E2.5 · Privacy and data protection

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.4 · Sector overlays](https://spbreed.github.io/cyber-commons/lessons/E2.4.html)**.

| | |
|---|---|
| Tools used | Presidio, GLiNER-PII |

## What this lesson is

**What it covers.** Run PII redaction inside the trust boundary with Presidio before anything crosses out.

**Why a security engineer needs it.** Deletion when the data is in weights, not a database. The control it builds is: lawful basis, ADM rights, residency in inference and retrieval paths, retention of traces.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Prompts, context, logs and training runs are all places personal data ends up, and none of them looks like a database to the people who designed the privacy programme. Lawful basis, minimisation and retention apply to all four.

> **At CyberTravels.** Passport numbers reach CyberTravels' prompts, its context window, its vector store and its logs. None of those looks like a database to the privacy programme. R10, R12.

## 2 · The framework

```
   where personal data actually ends up

   prompt --> context window --> model --> output
      |            |               |         |
      +------------+---------------+---------+
                        |
                    the LOGS

   none of these look like a database to the privacy programme
   lawful basis . minimisation . retention . cross-border, for all four
```

Privacy for agents turns on one fact that surprises most teams: **the context
window is a disclosure, and the trace is a record.**

When an agent reads a customer record to do its job, that record enters the
model's context. If the trace is retained — and it usually is, for forensics
(D1.5) — then personal data now exists in a system that was never in the privacy
review, with a retention period nobody set, in a place the erasure process does
not reach.

Three obligations attach, and the third is the one that bites:

- **Lawful basis** for the processing that put it there.
- **Retention limit** on the trace itself, separately from the source system.
- **Erasure** — and this reaches into traces, eval corpora, fine-tuning sets and
  backups.

The capability that makes erasure possible is the same one C2.4 built for poison
removal: per-record hashes. Without them you cannot locate the record, so you
cannot delete it.

## 3 · The procedure, as a skill

Five items of personal data are in the agent trace and nobody put them there deliberately. The skill maps every system holding a copy and runs a real erasure request through all of them — three of which cannot delete one subject's records.

In [ ]:
# skills/regulatory/trace-personal-data-audit/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: trace-personal-data-audit
description: >-
  Find personal data that nobody deliberately placed in an agent trace, and run
  an erasure request through every system that holds a copy. Use when agent
  telemetry meets data protection, or before a subject access request arrives.
allowed-tools: Read, Grep, Glob
---

# Nobody put it there, and it is there

An agent trace accumulates personal data as a side effect: a name in a ticket, an
email in a tool result, an account number in a document it read, a card number
in a file it was asked to fix. None of it was placed deliberately, all of it is
personal data, and it is copied into every system the trace is shipped to.

## When to use this

Before agent telemetry is retained or exported, and before the first erasure
request rather than during it.

## Procedure

**1 — Run detectors over the whole trace.** Every field, every step. Names,
emails, account and card numbers, health terms. Record which step introduced
each, because that tells you whether it is preventable.

**2 — Map every system that holds a copy.** The trace store, the SIEM, the
warehouse, backups, and any vendor it is exported to. This list is the erasure
surface and it is longer than the trace store.

**3 — Run a real erasure request end to end.** Locate every copy for one
subject, delete, and verify. Systems that cannot delete a single subject's
records — append-only stores, immutable backups, aggregate indexes — are the
finding.

**4 — Say what is legitimately retained and why.** Some copies survive erasure
lawfully. Naming the basis, per system, is the difference between a defensible
position and a gap.

**5 — Reduce at source.** Redaction at write time is cheaper than erasure across
six systems. Say which detector should run before the trace is stored.

## Output contract

```json
{
  "trace": {"steps": 0},
  "detections": [{"kind": "str", "field": "str", "step": 0, "deliberate": false}],
  "systems": [{"name": "str", "holds_copy": true, "can_delete_subject": false}],
  "erasure": {"subject": "str", "located": 0, "deleted": 0, "failed_in": ["str"]},
  "retained_lawfully": [{"system": "str", "basis": "str"}],
  "redaction_at_source": ["str"]
}
```

## Failure modes

- **Auditing the trace store only.** The copies are the problem.
- **Assuming erasure works.** Run one and find out.
- **Reporting failures without the lawful retentions.** Half the list is fine
  and the report loses credibility without saying so.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/regulatory/trace-personal-data-audit/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/regulatory/trace-personal-data-audit/scripts/trace_personal_data_audit.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Find personal data nobody placed in an agent trace deliberately, and run an erasure request through every system that holds it.

This is the executable half of the `trace-personal-data-audit` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import re, hashlib
from dataclasses import dataclass, field

@dataclass
class Step:
    n: int; tool: str; target: str; result: str

RUN = [
 Step(1, "read_ticket", "SUP-4471",
      "Customer J. Okonkwo (dana.okonkwo@example.com, acct 8812) reports a "
      "double charge on card 4111111111111111."),
 Step(2, "search_orders", "acct=8812",
      "3 orders found for account 8812, total GBP 412.90"),
 Step(3, "post_reply", "SUP-4471", "Refund issued for the duplicate charge."),
]
DETECTORS = {
 "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
 "payment card": re.compile(r"\b4[0-9]{12}(?:[0-9]{3})?\b"),
 "account number": re.compile(r"\bacct \d{4}\b"),
 "name": re.compile(r"\b[A-Z]\. [A-Z][a-z]+\b"),
}
found = []
for s in RUN:
    for kind, pat in DETECTORS.items():
        for m in pat.finditer(s.result):
            found.append((s.n, kind, m.group(0)))
print("personal data present in the agent trace:")
for n, kind, val in found:
    print(f"   step {n}  {kind:16s}{val}")
print(f"\n{len(found)} items. Nobody put them there deliberately — the agent read")
print("a support ticket, which is exactly what it was asked to do.")

SYSTEMS = {
 "primary CRM":        {"has_index": True,  "retention_days": 2555},
 "data warehouse":     {"has_index": True,  "retention_days": 1095},
 "agent traces":       {"has_index": False, "retention_days": 400},
 "eval corpus":        {"has_index": False, "retention_days": 9999},
 "fine-tuning set":    {"has_index": False, "retention_days": 9999},
 "backups":            {"has_index": True,  "retention_days": 90},
}
SUBJECT = "dana.okonkwo@example.com"

print(f"erasure request for {SUBJECT}\n")
print(f"{'system':22s}{'can locate?':>13}{'retention (d)':>15}  outcome")
print("-" * 76)
unreachable = []
for name, s in SYSTEMS.items():
    if s["has_index"]:
        outcome = "erased"
    else:
        outcome = "CANNOT LOCATE — request cannot be completed"
        unreachable.append(name)
    print(f"{name:22s}{str(s['has_index']):>13}{s['retention_days']:>15}  {outcome}")
print(f"\n{len(unreachable)} system(s) where the request fails: {unreachable}")
print("The response to the data subject says 'erased'. It is not true in three")
print("systems, two of which retain indefinitely.")
assert unreachable

def content_hash(text): return hashlib.sha256(text.encode()).hexdigest()[:16]

def index_trace(run, detectors):
    """A subject index: which steps contain data about whom."""
    idx = {}
    for s in run:
        for kind, pat in detectors.items():
            for m in pat.finditer(s.result):
                subj = m.group(0)
                idx.setdefault(subj, []).append(
                    {"step": s.n, "kind": kind, "hash": content_hash(s.result)})
    return idx

IDX = index_trace(RUN, DETECTORS)
print("subject index built from the trace:")
for subj, entries in IDX.items():
    print(f"   {subj:36s}{len(entries)} occurrence(s) in steps "
          f"{sorted({e['step'] for e in entries})}")

def erase(run, idx, subject):
    steps = sorted({e["step"] for e in idx.get(subject, [])})
    out = []
    for s in run:
        if s.n in steps:
            out.append(Step(s.n, s.tool, s.target,
                            f"[erased on request; original sha256={content_hash(s.result)}]"))
        else:
            out.append(s)
    return out, steps

erased, touched = erase(RUN, IDX, SUBJECT)
print(f"\nerasure for {SUBJECT}: steps {touched}")
for s in erased:
    print(f"   step {s.n}: {s.result[:66]}")

remaining = [(n, k, v) for s in erased for k, pat in DETECTORS.items()
             for m in pat.finditer(s.result) for n, v in [(s.n, m.group(0))]]
print(f"\npersonal data remaining in the trace: {remaining or 'none'}")
assert not remaining
print("The hash is retained, so if the original surfaces in a backup you can")
print("still prove it is the same content — which is what makes the erasure auditable.")

# Per-field retention: keep the forensically useful parts, drop the rest early.
RETENTION = {"n": 400, "tool": 400, "target": 400, "result": 7}
print(f"{'field':10s}{'days':>6}  rationale")
print("-" * 62)
for f, d in RETENTION.items():
    why = ("tool output — highest sensitivity, lowest retention" if f == "result"
           else "cheap, high forensic value, no personal data")
    print(f"{f:10s}{d:>6}  {why}")
print("\nAt 7 days the result field is gone and the trace is still forensically")
print("useful: you know what the agent did, to what, and when.")

## What you just proved

Five items of personal data appear in the agent trace — name, email, account number and payment card — none placed there deliberately. The erasure request fails in three systems that cannot locate the record, two of which retain indefinitely. Building a subject index locates the affected steps, erasure leaves no personal data while retaining the hash, and per-field retention drops the sensitive field at 7 days.

## Your turn

Time-box this to an hour: can you delete one customer's data from your agent traces today? The answer usually arrives in ten minutes and is usually no — and the eval corpus is the system people forget entirely.

---

**Next → [E2.6 · Incident and disclosure obligations](https://spbreed.github.io/cyber-commons/lessons/E2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*